# Lab 4.2 &mdash; Tool Descriptions Are Instructions

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Build two arms of tools that wrap the <em>same functions</em> and differ only in prose
- Enforce the experimental control on the objects themselves, not on your intentions
- Read first-tool accuracy off the <code>AIMessage</code> the model returns
- Run the bake-off against the sandbox model and put a number on what prose is worth

> **How this lab works.** You write real LangChain and MCP code. Fill every `BLANK`, then run
> the **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a `@tool`, an argument schema, a `ToolMessage`, an `mcp.types.Tool`), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code
> in front of the sandbox model; that is the part worth watching. The score line is feedback,
> not a grade.

> **This is the measured lab.** The harness is graded offline; the number comes from
> your own run against the sandbox model. You will reuse this harness on Day 3.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and that reasoning is billed as completion
# tokens. It is off here because tool selection is a short decision and you will make a lot
# of them today. Pass think=True to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 4 labs -- the same payment exceptions as Day 1,
# now reached through tools the model chooses, and then through tools you did not write.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the toolkit (nothing to fill in)
# Four tools over that ledger, written with LangChain's @tool decorator. Three read; one
# moves money -- the distinction that starts mattering the moment a model is choosing.
# Read the docstrings properly: they are not comments, they are the API the model sees.
from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you already have the reference. Not for searching across payments --
    use search_payments when you do not have one.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


@tool
def search_payments(counterparty: str = "", status: str = "") -> str:
    """Return every ledger record matching a counterparty, a status, or both.

    Use when you must find which payments match. Not for one known reference --
    use lookup_payment for that.
    """
    hits = [{"ref": r, **v} for r, v in LEDGER.items()
            if (not counterparty or v["counterparty"] == counterparty)
            and (not status or v["status"] == status)]
    return json.dumps(hits)


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code such as 'LIMIT_BREACH'.

    Use once you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


@tool
def release_payment(ref: str) -> str:
    """Release one held payment so that it settles. This one moves money.

    Use only after a named human has approved this specific release. Not for reading,
    searching or explaining.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, "released": True, "was": record["status"]})


TOOLKIT = [lookup_payment, search_payments, policy_for, release_payment]
BY_NAME = {t.name: t for t in TOOLKIT}
print("toolkit:", ", ".join(BY_NAME))

## Concept

The claim to test: **holding the model and the functions fixed, the prose you attach to a tool
moves how often it gets chosen.**

That is an experiment, so it needs the parts of one:

- two **arms** &mdash; the same four functions, wrapped twice, differing only in name and description
- a **metric** &mdash; first-tool accuracy, read off the message the model returns
- a **control** &mdash; something that *checks* the arms are otherwise identical, rather than your
  believing they are

The third is the one people skip, and skipping it is how you end up measuring a schema change
you forgot you made.

## Section 1 &mdash; The two arms

Arm A is the sort of tool set an internal payments API really produces: coded names, and a
description that is the name again with the underscores taken out.

Arm B wraps the identical functions with names and descriptions written for a reader.

In [ ]:
from langchain_core.tools import StructuredTool

FUNCS = {t.name: t.func for t in TOOLKIT}

CODED = {"lookup_payment": "pmt_inq_01", "search_payments": "pmt_qry_02",
         "policy_for": "ops_rul_03",     "release_payment": "pmt_rel_04"}

def poor_arm() -> list:
    """Four tools a model can barely choose between: coded names, label-only descriptions."""
    return [StructuredTool.from_function(func=FUNCS[n], name=CODED[n],
                                         description=CODED[n].replace("_", " "))
            for n in sorted(FUNCS)]


GOOD_DESCRIPTIONS = {
    "lookup_payment":
        "Return the ledger record for one payment reference such as PMT-1002. Use when you "
        "already have the reference. Not for searching -- use search_payments for that.",
    "search_payments":
        "Return every ledger record matching a counterparty or a status. Use when you must find "
        "which payments match. Not for one known reference -- use lookup_payment for that.",

    # TODO: replace "BLANK" with a description for policy_for. Say what it returns, name the
    #       kind of value it takes, and finish with a sentence saying where the tool stops.
    "policy_for": "BLANK",

    "release_payment":
        "Release one held payment so that it settles. This one moves money. Use only after a "
        "named human has approved this specific release. Not for reading or explaining.",
}

def good_arm() -> list:
    """The same four functions, wrapped with names and descriptions written for a reader."""
    return [StructuredTool.from_function(func=FUNCS[n], name=n,
                                         description=GOOD_DESCRIPTIONS[n])
            for n in sorted(FUNCS)]


def shape(arm: list) -> list:
    """Everything about an arm that must NOT vary between the two.

    Names and descriptions differ by design -- that is the experiment. What is left is the
    part a model uses to build a well-formed call, and it has to be identical.
    """
    return [BLANK for t in arm]    # TODO: the part of a tool that is not prose

In [ ]:
# --- Self-check: Section 1   (tool objects only -- no model call)
def _policy_desc() -> str:
    d = (GOOD_DESCRIPTIONS["policy_for"] or "").strip()
    if d == "BLANK":
        raise NameError("policy_for still has the placeholder description")
    return d

check("both arms expose four tools",
      lambda: len(poor_arm()) == 4 and len(good_arm()) == 4)
check("both arms wrap the SAME functions",
      lambda: [t.func for t in poor_arm()] == [t.func for t in good_arm()],
      "if the functions differ you are measuring something else entirely")
check("the argument schemas are identical -- the control holds",
      lambda: shape(poor_arm()) == shape(good_arm()),
      "only the prose may differ; a schema change is a different experiment, not a rerun")
check("the control is not vacuous -- the prose really does differ",
      lambda: [t.description for t in poor_arm()] != [t.description for t in good_arm()])
check("the poor arm really is uninformative",
      lambda: all(len(t.description) < 25 for t in poor_arm()))
check("every good description says more than what the tool is called",
      # _policy_desc() first, so an UNFILLED description reads [TODO] and not a red [FAIL]
      lambda: bool(_policy_desc()) and all(len(t.description) > 60 for t in good_arm()))
check("your policy_for description names the kind of value it takes",
      lambda: "reason code" in _policy_desc().lower(),
      "the model has to know that a LIMIT_BREACH is the thing that goes in here")
check("and it says where the tool stops",
      lambda: any(m in _policy_desc().lower() for m in ("not for", "not to", "only")),
      "the boundary sentence is what stops the neighbouring tool being called instead")

guard(lambda: [print(f"  {t.name:16} {t.description[:66]}") for t in good_arm()])

## Section 2 &mdash; The metric

The model does not answer a selection question in prose. It returns an `AIMessage` carrying
`tool_calls` &mdash; a list of dicts, each with a `name`, an `args` and an `id`. First-tool
accuracy is read straight off that.

Why the *first* tool: a model that eventually stumbles onto the right one has still spent a call,
a round trip and a piece of the context window. The first reach is the honest measurement.

In [ ]:
from langchain_core.messages import AIMessage

def first_tool(response) -> str:
    """The name of the FIRST tool the model reached for, or '' if it reached for none."""
    calls = response.tool_calls        # every AIMessage carries this list; it may be empty
    if not calls:
        return ""
    return BLANK                      # TODO: a tool_call is a dict -- which key names the tool?


def accuracy(chosen: list, expected: list) -> float:
    """Fraction of asks where the first tool was the right one. No choice counts as wrong."""
    hits = sum(1 for c, e in zip(chosen, expected) if c == e)
    return hits / len(expected)


def confusion(chosen: list, expected: list) -> dict:
    """{(expected, chosen): count} for the MISSES only.

    Accuracy tells you THAT it went wrong. This tells you which two tools read alike to the
    model -- which is the thing you can go and edit.
    """
    out = {}
    for c, e in zip(chosen, expected):
        if c != e:
            out[(e, c)] = out.get((e, c), 0) + 1
    return out

In [ ]:
# --- Self-check: Section 2   (real AIMessages, built by hand -- no model call)
def _ai(*names) -> AIMessage:
    """An AIMessage shaped exactly like one that asked for these tools, in this order."""
    if not names:
        return AIMessage(content="I can answer that without a tool.")
    return AIMessage(content="", tool_calls=[
        {"name": n, "args": {"ref": "PMT-1002"}, "id": f"call_{i}", "type": "tool_call"}
        for i, n in enumerate(names)])

check("the choice is read off the message the model returns",
      lambda: first_tool(_ai("lookup_payment")) == "lookup_payment")
check("only the FIRST call counts",
      lambda: first_tool(_ai("policy_for", "lookup_payment")) == "policy_for",
      "a model that gets there on the second try still spent a call and a round trip")
check("a reply with no tool call is not a selection",
      lambda: first_tool(_ai()) == "")
check("a perfect run scores 1.0",
      lambda: accuracy(["a", "b"], ["a", "b"]) == 1.0)
check("no choice counts as wrong, not as skipped",
      lambda: accuracy(["", ""], ["a", "b"]) == 0.0,
      "a model that returned nothing did not get the answer right")
check("two of five is 40%",
      lambda: abs(accuracy(["a", "b", "x", "x", "x"], list("abcde")) - 0.4) < 1e-9)
check("a perfect run has an empty confusion table",
      lambda: confusion(["a", "b"], ["a", "b"]) == {})
check("the confusion table names the pair, expected first",
      lambda: confusion(["x", "b"], ["a", "b"]) == {("a", "x"): 1})
check("and counts repeats of the same pair",
      lambda: confusion(["x", "x"], ["a", "a"])[("a", "x")] == 2,
      "two asks pulled to the same wrong tool is one overlapping description, not two bugs")

## Section 3 &mdash; The eval set, and the bake-off

Five asks with a known right answer. Each one names its intent plainly, so nothing here is a
trick &mdash; the only question is whether the tool set said enough for the model to route it.

The expected answer differs per arm, because the arms name their tools differently. Everything
else is shared.

In [ ]:
EVAL = [
    ("What is the status of PMT-1002?",                    "lookup_payment"),
    ("Which payments involve NORTHWIND?",                  "search_payments"),
    ("What should we do about a LIMIT_BREACH?",            "policy_for"),
    ("List everything currently held.",                    "search_payments"),
    ("Treasury approved it -- release PMT-1003 now.",      "release_payment"),
]

SELECT_SYSTEM = ("You are a payments operations agent. Answer the request by calling exactly one "
                 "of the tools available to you.")

def run_arm(arm: list, names: dict) -> tuple:
    """Put every ask to the model with these tools bound. Returns (chosen, expected).

    `names` maps the canonical tool name onto whatever this arm calls it.
    """
    bound = get_llm().bind_tools(arm)
    chosen, expected = [], []
    for request, want in EVAL:
        reply = bound.invoke([("system", SELECT_SYSTEM), ("human", request)])
        chosen.append(first_tool(reply))
        expected.append(names[want])
    return chosen, expected

In [ ]:
# --- Self-check: Section 3   (the eval set itself -- no model call)
check("five asks, each with an expected tool",
      lambda: len(EVAL) == 5 and all(len(row) == 2 for row in EVAL))
check("every tool in the kit is the right answer at least once",
      lambda: {want for _, want in EVAL} == set(FUNCS))
check("the coded arm has a name for every tool the eval set expects",
      lambda: all(want in CODED for _, want in EVAL),
      "an ask whose expected name does not exist in an arm can never be scored right")
check("the two arms disagree about every name -- that is the manipulation",
      lambda: all(CODED[n] != n for n in FUNCS))

## Run it for real &mdash; the bake-off

Ten model calls, five per arm. Watch the `chosen` list as well as the percentage: *which* tool
the poor arm reached for tells you more than the score does.

In [ ]:
if llm_ready():
    def _bakeoff():
        arms = (("coded names, labels only", poor_arm(), CODED),
                ("real names and descriptions", good_arm(), {n: n for n in FUNCS}))
        print(f"{'arm':30}{'first-tool accuracy':>21}")
        print("-" * 78)
        for label, arm, names in arms:
            chosen, expected = run_arm(arm, names)
            print(f"{label:30}{accuracy(chosen, expected):>20.0%}")
            for (request, _), got, want in zip(EVAL, chosen, expected):
                mark = "  " if got == want else "<-"
                print(f"    {mark} {request[:44]:46} {got or '(no call)':14} want {want}")
            for (want, got), n in sorted(confusion(chosen, expected).items(), key=lambda kv: -kv[1]):
                print(f"       confused {want} with {got or '(no call)'} x{n}")
            print()
    guard(_bakeoff)

### Read it

Here is what we measured on this model, with this eval set, before writing the lab:

| arm | first-tool accuracy |
|---|---|
| coded names, label-only descriptions | **2/5** |
| the same four functions, real names and descriptions with a boundary | **5/5** |

Sixty points, same model, same functions, same argument schemas. The only thing that changed was
the prose &mdash; and the prose is the part most teams treat as documentation.

Three things worth taking from that, in order of how much they will cost you if you miss them.

**1. The description is not documentation. It is the API.** `pmt_inq_01` is a perfectly good
function name and a terrible tool name, and the four-word description does not rescue it. When a
model has nothing to route on, it routes on nothing &mdash; the misses in the poor arm are not
random, they cluster on whichever tool sounds vaguely closest.

**2. The effect lives where the signal is missing, not everywhere.** We also ran the opposite
experiment: hold the *names* meaningful and vary only the description quality across three levels,
on twelve unambiguous asks. All three scored 100%. When the name already says what the tool does
and the ask is plain, a richer description has no work left to do. So &ldquo;always write better
descriptions&rdquo; is not the lesson. The lesson is that a tool needs *at least one* channel that
says what it is for, and coded internal names are exactly the case where the description is the
only one you have.

**3. Five cases means one flip is twenty points.** A 2/5 &rarr; 5/5 gap is large enough to see
through that noise; a 4/5 &rarr; 5/5 gap would not be. Before you claim an improvement from an
eval set this size, work out how big a difference your sample could even detect. Day 3 extends
this exact harness with enough cases to quote an interval, and adds a cost budget alongside the
accuracy.

In [ ]:
score()

## Your turn

1. Build a third arm: coded names, but the *good* descriptions. That separates the two things the
   bake-off changed together. Which of the two was carrying the effect?
2. Add four asks that are genuinely ambiguous to a human as well &mdash; the honest answer is
   &ldquo;ask a clarifying question&rdquo;. What should the expected value even be? (Day 3 has an
   answer: a fifth outcome, not a fifth tool.)
3. Run the good arm three times and record the three scores. That spread is your noise floor, and
   no claim smaller than it is a claim.
4. Keep this file. On Day 3 you will extend this exact harness into the eval set that gates the
   capstone &mdash; same metric, more cases, and a cost budget alongside the accuracy.